# Evaluation Framework

This notebook reproduces the evaluation pipeline described in the paper. The original data ingestion has been replaced with a small in-memory mock dataset so the notebook can be executed standalone for demonstration and reproducibility. All identifiers (names, streets, postal codes) are randomly chosen and do not correspond to real individuals or locations.

In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd, numpy as np
import ast
from collections import defaultdict
from nervaluate import Evaluator

In [ ]:
# Label schema used in the paper (Table: Category groups and labels).
# The TRANSLATION_DICT maps the English labels used by the evaluator
# to the Dutch labels used in downstream reporting.

TRANSLATION_DICT = {
    "NAME": "NAAM",
    "NAME_GIVEN": "NAAM_VOORNAAM",
    "NAME_FAMILY": "NAAM_ACHTERNAAM",
    "DOB": "GEBOORTEDATUM",
    "LOCATION": "LOCATIE",
    "LOCATION_ADDRESS_STREET": "LOCATIE_ADRES_STRAAT",
    "LOCATION_CITY": "LOCATIE_STAD",
    "LOCATION_ZIP": "LOCATIE_POSTCODE",
    "PHONE_NUMBER": "TELEFOONNUMMER",
    "EMAIL_ADDRESS": "E-MAILADRES",
    "BANK_ACCOUNT": "BANKREKENING"
}

In [ ]:
# Mock dataset for demonstration.
# The original data ingestion has been removed because the underlying
# corpus is proprietary. This synthetic transcript reproduces the
# example shown in the paper's fault-analysis figure.

MOCK_TEXT = (
    "agent goedemiddag u spreekt met de klantenservice waarmee kan ik u helpen "
    "ja goedemiddag u spreekt met Jon van Dough uit Leeuwarden "
    "ik bel over mijn abonnement dat sinds vorige week niet meer werkt "
    "agent dat vind ik vervelend om te horen kunt u mij uw adres bevestigen "
    "ja natuurlijk mijn adres is Hoogweidelaan 84 1111 XA Leeuwarden "
    "kunt u het daar naartoe sturen"
)

def _locate(text, needle, start=0):
    idx = text.index(needle, start)
    return idx, idx + len(needle)

c1_start, c1_end = _locate(MOCK_TEXT, "Jon van Dough")
city1_start, city1_end = _locate(MOCK_TEXT, "Leeuwarden")
street_start, street_end = _locate(MOCK_TEXT, "Hoogweidelaan 84")
c2_start, c2_end = _locate(MOCK_TEXT, "1111 XA")
city2_start, city2_end = _locate(MOCK_TEXT, "Leeuwarden", city1_end)

# Ground truth annotations
true_labels = [
    {"text": "Jon van Dough", "label": "NAME",
     "start": c1_start, "end": c1_end},
    {"text": "Leeuwarden", "label": "LOCATION_CITY",
     "start": city1_start, "end": city1_end},
    {"text": "Hoogweidelaan 84", "label": "LOCATION_ADDRESS_STREET",
     "start": street_start, "end": street_end},
    {"text": "1111 XA", "label": "LOCATION_ZIP",
     "start": c2_start, "end": c2_end},
    {"text": "Leeuwarden", "label": "LOCATION_CITY",
     "start": city2_start, "end": city2_end},
]

# Predictions from the model under evaluation.
# Card 1: boundaries match ground truth, label differs (NAME_GIVEN vs NAME).
# Card 2: boundaries and label match ground truth.
pred_labels = [
    {"text": "Jon van Dough", "label": "NAME_GIVEN",
     "start": c1_start, "end": c1_end},
    {"text": "1111 XA", "label": "LOCATION_ZIP",
     "start": c2_start, "end": c2_end},
]

evaluation_data_prepped_df = pd.DataFrame({
    "text": [MOCK_TEXT],
    "privateai_labels": [str(pred_labels)],
    "true_labels": [str(true_labels)]
})
evaluation_data_prepped_df

In [ ]:
# Convert stringified objects to Python objects
evaluation_data_prepped_df['privateai_labels'] = evaluation_data_prepped_df['privateai_labels'].apply(ast.literal_eval)
evaluation_data_prepped_df['true_labels'] = evaluation_data_prepped_df['true_labels'].apply(ast.literal_eval)

In [ ]:
def filter_labels(labels):
    return [label for label in labels if label['label'] in TRANSLATION_DICT.keys()]

evaluation_data_prepped_df['true_labels'] = evaluation_data_prepped_df['true_labels'].apply(filter_labels)

In [ ]:
print(set([label['label'] for sublist in evaluation_data_prepped_df['true_labels'] for label in sublist]))

In [ ]:
labels = evaluation_data_prepped_df.explode('true_labels')['true_labels'].apply(lambda x: x['label'])
labels.value_counts()

In [ ]:
labels = evaluation_data_prepped_df.explode('privateai_labels')['privateai_labels'].apply(lambda x: x['label'])
labels.value_counts()

In [ ]:
def run_evaluation(evaluation_df):
    """Run nervaluate evaluator and return the results."""
    gt = evaluation_df['true_labels'].to_list()
    pred = evaluation_df['privateai_labels'].to_list()

    evaluator = Evaluator(gt, pred, tags=list(TRANSLATION_DICT.keys()), loader='dict')
    results = evaluator.evaluate()
    return evaluator, results

evaluator, results = run_evaluation(evaluation_data_prepped_df)

In [ ]:
def as_dict(obj):
    """Convert dataclass-like objects to plain dicts."""
    if hasattr(obj, "__dict__"):
        return {k: v for k, v in vars(obj).items()}
    return obj


def build_overall_metrics(raw):
    """Convert nervaluate evaluation metrics to pandas DataFrame."""
    overall_records = []
    for metric_set, result in raw["overall"].items():
        d = as_dict(result).copy()
        d["policy"] = metric_set
        overall_records.append(d)
    df_overall = pd.DataFrame(overall_records).set_index("policy").sort_index()
    return df_overall


def build_entity_metrics(raw):
    """Convert per-entity nervaluate evaluation metrics to pandas DataFrame."""
    entity_rows = []
    for entity, metrics in raw["entities"].items():
        for metric_set, result in metrics.items():
            d = as_dict(result).copy()
            d["entity"] = entity
            d["policy"] = metric_set
            entity_rows.append(d)
    df_entities = pd.DataFrame(entity_rows).set_index(["entity", "policy"]).sort_index()
    return df_entities


def build_overall_indices(raw):
    """Convert overall indices to pandas DataFrame."""
    overall_idx_rows = []
    for metric_set, idx in raw["overall_indices"].items():
        d = as_dict(idx).copy()
        d["policy"] = metric_set
        overall_idx_rows.append(d)
    df_overall_indices = pd.DataFrame(overall_idx_rows).set_index("policy").sort_index()
    return df_overall_indices


def build_fixed_entity_indices(fixed_entity_dict):
    """Convert fixed nervaluate evaluation indices to pandas DataFrame."""
    label_frames = {}
    for label, policy in fixed_entity_dict.items():
        df = pd.DataFrame.from_dict(policy)
        label_frames[label] = df
    full_df = pd.concat(label_frames)
    full_df = full_df.reset_index().rename(columns={'level_0': 'label', 'level_1': 'measure'})
    return full_df


def build_entity_indices(raw):
    """Convert per-entity nervaluate indices to pandas DataFrame."""
    entity_idx_rows = []
    for entity, metrics in raw["entity_indices"].items():
        for metric_set, idx in metrics.items():
            d = as_dict(idx).copy()
            d["entity"] = entity
            d["policy"] = metric_set
            entity_idx_rows.append(d)
    df_entity_indices = pd.DataFrame(entity_idx_rows).set_index(["entity", "policy"]).sort_index()
    return df_entity_indices

In [ ]:
results['overall_indices']

In [ ]:
def group_spans_by_label(data, true, pred, categories=None, label_key="label"):
    """Regroup spans from index buckets by their label."""
    if categories is None:
        categories = [k for k in data.keys() if k.endswith("_indices")]

    out = defaultdict(lambda: {c: [] for c in categories})

    for cat in categories:
        for span_idx in data.get(cat, []):
            if cat == 'missed_indices':
                span = true[span_idx[0]][span_idx[1]]
            else:
                span = pred[span_idx[0]][span_idx[1]]
            label = span.get(label_key)
            if label is None:
                continue
            label = str(label).strip()
            out[label][cat].append(span_idx)

    return dict(out)

strict_indices_dict = results['overall_indices']['strict'].__dict__
exact_indices_dict = results['overall_indices']['exact'].__dict__
partial_indices_dict = results['overall_indices']['partial'].__dict__
enttype_indices_dict = results['overall_indices']['ent_type'].__dict__

entity_indices = {}
entity_indices['strict'] = group_spans_by_label(strict_indices_dict, evaluation_data_prepped_df['true_labels'], evaluation_data_prepped_df['privateai_labels'])
entity_indices['exact'] = group_spans_by_label(exact_indices_dict, evaluation_data_prepped_df['true_labels'], evaluation_data_prepped_df['privateai_labels'])
entity_indices['partial'] = group_spans_by_label(partial_indices_dict, evaluation_data_prepped_df['true_labels'], evaluation_data_prepped_df['privateai_labels'])
entity_indices['ent_type'] = group_spans_by_label(enttype_indices_dict, evaluation_data_prepped_df['true_labels'], evaluation_data_prepped_df['privateai_labels'])

In [ ]:
EVAL_MODES = ("strict", "partial", "ent_type", "exact")
BUCKETS = ("correct_indices", "incorrect_indices", "partial_indices",
           "missed_indices", "spurious_indices")

def convert_eval_format(src: dict) -> dict:
    """Convert { mode: { ENTITY: {bucket: [...]} } } into { ENTITY: { mode: {bucket: [...]} } }."""
    all_entities = set()
    for mode_dict in src.values():
        all_entities.update(mode_dict.keys())

    out = {
        entity: {
            mode: {bucket: [] for bucket in BUCKETS}
            for mode in EVAL_MODES
        }
        for entity in all_entities
    }

    for mode, mode_dict in src.items():
        for entity, buckets in mode_dict.items():
            if entity not in out:
                out[entity] = {m: {b: [] for b in BUCKETS} for m in EVAL_MODES}
            if mode not in out[entity]:
                out[entity][mode] = {b: [] for b in BUCKETS}
            for b in BUCKETS:
                out[entity][mode][b] = list(buckets.get(b, []))
    return out

fixed_entity_indices = convert_eval_format(entity_indices)

In [ ]:
entity_indices_df = build_fixed_entity_indices(fixed_entity_indices)
entity_indices_df

In [ ]:
# Build metrics DataFrames
overall_metrics = build_overall_metrics(results)
entity_metrics = build_entity_metrics(results)

# Add multi-index column to overall metrics so they concat to entity metrics
entity_col = ['overall' for _ in range(overall_metrics.shape[0])]
overall_metrics.insert(0, column='entity', value=entity_col)
overall_metrics = overall_metrics.set_index('entity', append=True)
overall_metrics = overall_metrics.swaplevel('policy', 'entity')

# Combine metrics
full_metrics = pd.concat([overall_metrics, entity_metrics]).reset_index()
full_metrics

In [ ]:
entity_indices = build_entity_indices(results)

def convert_indices_nervis(entity_indices):
    """Pivot indices DataFrame to make it compatible with NERvis."""
    conv_df = entity_indices.stack().reset_index()
    conv_df.columns = ['entity', 'policy', 'measure', 'value']
    conv_df = conv_df.pivot_table(
        index=['entity', 'measure'],
        columns='policy',
        values='value',
        aggfunc=lambda x: x)
    conv_df = conv_df.reset_index()
    conv_df.columns.name = None
    return conv_df

entity_indices = convert_indices_nervis(entity_indices)
entity_indices

In [ ]:
# Final outputs
privateai_evaluation_metrics_df = full_metrics
privateai_indices_df = entity_indices_df

print("Evaluation metrics:")
display(privateai_evaluation_metrics_df)
print("Indices:")
display(privateai_indices_df)